In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm=ChatOpenAI(model="gpt-4o-mini")

In [4]:
class jokeState(TypedDict):
    topic:str
    joke: str
    explanation: str

In [19]:
def generate_joke(State: jokeState):
    topic=State["topic"]
    prompt=f'generate a joke on the topic given-{topic}'
    response=llm.invoke(prompt).content
    
    return {"joke": response}

In [20]:
def joke_explanation(State: jokeState)->jokeState:
    joke=State["joke"]
    prompt=f'explain the joke {joke}'
    explanation=llm.invoke(prompt).content
    
    return {"explanation": explanation}

In [21]:
graph=StateGraph(jokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("joke_explanation",joke_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "joke_explanation")
graph.add_edge("joke_explanation", END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [22]:
initial_state={"topic": "AI"}
final_state=workflow.invoke(initial_state, config={"configurable":{"thread_id": "thread-1"}})

In [23]:
final_state

{'topic': 'AI',
 'joke': 'Why did the AI break up with its calculator?\n\nBecause it felt like it was just adding to the drama!',
 'explanation': 'This joke plays on a pun involving the word "adding." In a relationship context, "adding to the drama" means to contribute to or enhance complications and emotional turmoil. The humor comes from the clever wordplay, as calculators perform arithmetic operations, primarily addition. \n\nSo, when the AI breaks up with the calculator, it’s not just a literal interpretation of a mathematical operation; it’s a play on the idea that the relationship was causing unnecessary complications, akin to how adding numbers can exceed desired simplicity. The joke is a lighthearted way of portraying how even an AI, which operates on logic, might find a relationship burdensome—much like humans might when stresses accumulate.'}

In [27]:
initial_state2={"topic": "you"}
final_state2=workflow.invoke(initial_state2, config={"configurable":{"thread_id": "thread-2"}})
final_state2

{'topic': 'you',
 'joke': 'Why did the AI go to therapy? \n\nBecause it had too many "processing" issues!',
 'explanation': 'The joke "Why did the AI go to therapy? Because it had too many \'processing\' issues!" plays on a pun involving the word "processing." \n\nIn the context of AI, "processing" refers to how the artificial intelligence handles and analyzes data. When AI is said to have processing issues, it typically means it\'s struggling with its computations or decision-making abilities.\n\nOn the other hand, in a more human context, "processing issues" can refer to emotional or psychological challenges a person might have—essentially how they manage their feelings, thoughts, or experiences. \n\nBy combining these two meanings, the joke humorously suggests that an AI, like a human, might need therapy for its "processing" problems, creating a clever wordplay that highlights the human-like traits we sometimes attribute to AI. It effectively blends a technological term with a relat

In [32]:
config = {"configurable": {"thread_id": "thread-1"}}

workflow.get_state(config)

StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the AI break up with its calculator?\n\nBecause it felt like it was just adding to the drama!', 'explanation': 'This joke plays on a pun involving the word "adding." In a relationship context, "adding to the drama" means to contribute to or enhance complications and emotional turmoil. The humor comes from the clever wordplay, as calculators perform arithmetic operations, primarily addition. \n\nSo, when the AI breaks up with the calculator, it’s not just a literal interpretation of a mathematical operation; it’s a play on the idea that the relationship was causing unnecessary complications, akin to how adding numbers can exceed desired simplicity. The joke is a lighthearted way of portraying how even an AI, which operates on logic, might find a relationship burdensome—much like humans might when stresses accumulate.'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f163b18-3594-695